In [144]:
import networkx as nx
from bp.world import random_grid_world, Scenario

In [145]:
world = random_grid_world(
    rows=4,
    cols=4,
    demands={
        ((0, 0), (3, 3)): 10,
        ((1, 0), (1, 3)): 5,
    },
    seed=0,
)
G = world.network.graph

print(f"{world.total_population=}")
print(f"{world.total_demand((0, 0), (3, 3))=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")

world.total_population=15
world.total_demand((0, 0), (3, 3))=10
world.total_demand((1, 0), (1, 3))=5


In [146]:
from itertools import pairwise
def edge_path(path):
    """Given a path consisting of vertices, return the corresponding edge-path"""
    return list(pairwise(path)) # list for state safety, can remove if careful though

In [147]:
import numpy as np

def debug_scenario(realized, omega, volumes):
    # (volume - 1) b/c need to recalculate b/c of off-by-one-error
    active_travel_time = world.network.actual_travel_times({a: volume - 1 for a, volume in volumes.items()})
    active_arcs = {a: t for a, t in active_travel_time.items() if volumes[a] > 0}

    travel_time = sum(t * volumes[a] for a, t in active_arcs.items())
    average_travel_time = travel_time / world.total_population
    print(f"{travel_time=:.2f}")
    print(f"{average_travel_time=:.2f}")

    nominal_arc_time = sum(omega.travel_time[a] for a in active_arcs)
    realized_arc_time = sum(active_arcs.values())
    average_arc_time = realized_arc_time / len(active_arcs)
    arc_time_increase = realized_arc_time - nominal_arc_time
    average_arc_time_increase = arc_time_increase / len(active_arcs)
    print(f"{average_arc_time=:.2f} (along arcs w/ non-zero flow/volume)")
    print(f"{average_arc_time_increase=:.2f} (+{arc_time_increase / nominal_arc_time * 100:.2f}%, overall, not per-arc)")

    flow = np.array([volumes.get(arc, 0) for arc in world.ordered_arcs], dtype=float)
    capacity_used = flow / world.network._arc_array("capacity") * 100
    active_capacity_used = capacity_used[capacity_used > 0]
    average_capacity_used = np.average(active_capacity_used)
    most_capacity_used = np.max(capacity_used)
    print(f"{most_capacity_used=:.2f}%")
    print(f"{average_capacity_used=:.2f}%")

    # should sort then bin-search for `congested` b/c sorting but copy paste lazy
    congested = sum(active_arcs[a] > omega.travel_time[a] + 1e-9 for a in active_arcs)
    print(f"arcs whose travel time rose under congestion: {congested}/{len(world.A)}")

    most_congested = sorted(((a, (active_arcs[a] - omega.travel_time[a]) / omega.travel_time[a]) for a in active_arcs), reverse=True, key=lambda x: x[1])
    for i, ((orig, dest), delta) in enumerate(most_congested[:congested], 1):
        print(f"{i}: {orig}->{dest}: +{delta * 100:.2f}%")

In [148]:
routes = {
    plr: edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight="travel_time"))
    for plr in world.individuals
}

volumes = {arc: 0 for arc in world.A}
for route in routes.values():
    for arc in route:
        volumes[arc] += 1

realized_concurrent = world.realize(routes)
omega_concurrent = Scenario.from_world("nominal", world)

print("CONCURRENT:")
debug_scenario(realized_concurrent, omega_concurrent, volumes)

CONCURRENT:
travel_time=19406.81
average_travel_time=1293.79
average_arc_time=216.98 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=211.30 (+3722.65%, overall, not per-arc)
most_capacity_used=780.45%
average_capacity_used=361.33%
arcs whose travel time rose under congestion: 9/48
1: (2, 3)->(3, 3): +36513.31%
2: (0, 3)->(1, 3): +14975.04%
3: (0, 0)->(0, 1): +5266.48%
4: (1, 3)->(2, 3): +2611.71%
5: (0, 1)->(0, 2): +1877.45%
6: (0, 2)->(0, 3): +228.03%
7: (1, 2)->(1, 3): +44.31%
8: (1, 0)->(1, 1): +7.43%
9: (1, 1)->(1, 2): +6.82%


In [149]:
def dynamic_weight(orig, dest, _):
    return current_travel_times[(orig, dest)]

volumes = {arc: 0 for arc in world.A}
routes = {}
for plr in world.individuals:
    current_travel_times = world.network.actual_travel_times(volumes)
    route = edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight=dynamic_weight))
    routes[plr] = route
    for arc in route:
        volumes[arc] += 1

realized_sequential = world.realize(routes)
omega_sequential = Scenario.from_world("nominal", world)

print("SEQUENTIAL:")
debug_scenario(realized_sequential, omega_sequential, volumes)

SEQUENTIAL:
travel_time=568.21
average_travel_time=37.88
average_arc_time=7.34 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=1.26 (+20.64%, overall, not per-arc)
most_capacity_used=312.18%
average_capacity_used=115.66%
arcs whose travel time rose under congestion: 19/48
1: (2, 3)->(3, 3): +450.78%
2: (1, 0)->(1, 1): +69.66%
3: (0, 0)->(0, 1): +65.02%
4: (1, 2)->(1, 3): +44.31%
5: (0, 0)->(1, 0): +39.79%
6: (2, 2)->(3, 2): +28.43%
7: (3, 2)->(3, 3): +24.84%
8: (0, 1)->(0, 2): +23.18%
9: (2, 1)->(2, 2): +11.78%
10: (1, 1)->(1, 2): +6.82%
11: (2, 0)->(2, 1): +6.55%
12: (0, 3)->(1, 3): +2.28%
13: (0, 2)->(1, 2): +0.49%
14: (1, 0)->(2, 0): +0.45%
15: (1, 1)->(2, 1): +0.42%
16: (1, 3)->(2, 3): +0.40%
17: (1, 2)->(2, 2): +0.27%
18: (2, 2)->(2, 3): +0.12%
19: (0, 2)->(0, 3): +0.03%
